In [1]:
from tensorflow import keras
import urllib.request
import zipfile
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import random
import os
import urllib.request
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.inception_v3 import InceptionV3
from tensorflow.keras.layers import Flatten, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Nadam
from tensorflow.keras.callbacks import ModelCheckpoint
import pandas as pd
from sklearn.model_selection import train_test_split
import librosa

# Train InceptionV3

In [2]:
base = '../processed_data'
features_type = os.listdir(base)

for feature in features_type:
    base_dir = os.path.join(base, feature)

    filepaths = []
    labels = []

    for label in os.listdir(base_dir):
        class_dir = os.path.join(base_dir, label)
        if os.path.isdir(class_dir):
            for file in os.listdir(class_dir):

                filepaths.append(os.path.join(class_dir, file))
                labels.append(label)

    df = pd.DataFrame({
        'filename': filepaths,
        'class': labels
    })

    df.drop(0, axis=0, inplace=True)

    train_df, test_df = train_test_split(
        df,
        test_size=0.2,
        stratify=df['class'],
        random_state=42
    )

    datagen = ImageDataGenerator(rescale=1./255)

    train_generator = datagen.flow_from_dataframe(
        train_df,
        x_col='filename',
        y_col='class',
        target_size=(570, 370),
        class_mode='categorical',
        batch_size=16
    )

    test_generator = datagen.flow_from_dataframe(
        test_df,
        x_col='filename',
        y_col='class',
        target_size=(570, 370),
        class_mode='categorical',
        batch_size=16,
        shuffle=False
    )

    pre_trained_model = InceptionV3(
        input_shape=(570, 370, 3),
        include_top=False,
        weights='imagenet'
    )

    for layer in pre_trained_model.layers:
        layer.trainable = False

    last_output = pre_trained_model.get_layer('mixed7').output

    x = Flatten()(last_output)
    x = Dense(256, activation='relu')(x)
    x = Dense(128, activation='relu')(x)
    x = Dense(64, activation='relu')(x)
    x = Dropout(0.2)(x)
    x = Dense(32, activation='relu')(x)
    x = Dropout(0.2)(x)

    num_classes = len(train_generator.class_indices)
    x = Dense(num_classes, activation='softmax')(x)

    model = Model(pre_trained_model.input, x)

    model.compile(
        optimizer=Nadam(learning_rate=1e-4),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    checkpoint = ModelCheckpoint(f"checkpoints/model_{feature}.h5", save_best_only=True)

    history = model.fit(
        train_generator,
        epochs=10,
        validation_data=test_generator,
        callbacks=[checkpoint]
    )

Found 671 validated image filenames belonging to 7 classes.
Found 168 validated image filenames belonging to 6 classes.
Epoch 1/10
12/42 ━━━━━━━━━━━━━━━━━━━━ 1:49 4s/step - accuracy: 0.2853 - loss: 1.7955

KeyboardInterrupt: 

# Siamese network

In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
from keras.preprocessing.image import ImageDataGenerator
import random
# Função para criar a arquitetura da Rede Siamesa
def create_siamese_model(input_shape):
    input = layers.Input(input_shape)

    # Arquitetura da sub-rede (CNN simples)
    x = layers.Conv2D(64, (5,5), activation='tanh')(input)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(128, (3,3), activation='sigmoid')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(128, (3,3), activation='sigmoid')(x)
    x = layers.MaxPooling2D()(x)
    x = layers.Conv2D(128, (3,3), activation='sigmoid')(x)
    x = layers.Flatten()(x)
    x = layers.Dense(16, activation='sigmoid')(x)

    model = models.Model(input, x)
    return model

# Definir a função de distância entre as duas saídas
def euclidean_distance(vectors):
    (featA, featB) = vectors
    sum_squared = tf.reduce_sum(tf.square(featA - featB), axis=1, keepdims=True)
    return tf.sqrt(tf.maximum(sum_squared, tf.keras.backend.epsilon()))

# Input shape (28x28, como no dataset Omniglot)
input_shape = (300, 300, 3)

# Criação do modelo siamesa
base_network = create_siamese_model(input_shape)

# Definir as duas entradas
input_a = layers.Input(shape=input_shape)
input_b = layers.Input(shape=input_shape)

# Extração das características para as duas entradas
feat_a = base_network(input_a)
feat_b = base_network(input_b)

# Distância entre as saídas das redes
distance = layers.Lambda(euclidean_distance)([feat_a, feat_b])

# Modelo Siamesa completo
model = models.Model(inputs=[input_a, input_b], outputs=distance)

# Compilar o modelo
model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])

2024-10-02 09:39:29.430267: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2024-10-02 09:39:29.629143: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-10-02 09:39:30.803535: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT
2024-10-02 09:39:34.263938: E tensorflow/compiler/xla/stream_executor/cuda/cuda_driver.cc:266] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected


In [2]:
def generate_image_pairs(generator, num_pairs=1000):
    """
    Gera pares de imagens e os rótulos correspondentes, onde:
    - generator: Gerador de imagens (como o gerado pelo ImageDataGenerator)
    - num_pairs: Número de pares que você quer gerar
    """
    pairs = []
    labels = []

    X, y = generator.next()  # Carrega um batch de imagens e rótulos
    n_classes = len(np.unique(y))  # Número de classes no dataset
    class_indices = [np.where(y == i)[0] for i in range(n_classes)]  # Índices por classe

    for _ in range(num_pairs):
        if random.choice([True, False]):
            # Mesma classe
            class_id = random.randint(0, n_classes - 1)
            idx1, idx2 = random.sample(list(class_indices[class_id]), 2)
            pairs.append([X[idx1], X[idx2]])
            labels.append(1)  # Mesmo rótulo
        else:
            # Classes diferentes
            class_id1, class_id2 = random.sample(range(n_classes), 2)
            idx1 = random.choice(class_indices[class_id1])
            idx2 = random.choice(class_indices[class_id2])
            pairs.append([X[idx1], X[idx2]])
            labels.append(0)  # Classes diferentes

    return np.array(pairs), np.array(labels)

# Gerando 1000 pares de imagens a partir do diretório de treinamento
# X_pairs, y_pairs = generate_image_pairs(train_generator, num_pairs=1000)

In [3]:
training_dir = 'database_images/test'
test_dir = 'database_images/train'

train_datagen = ImageDataGenerator(
    rescale = 1/255,
    #rotation_range=45,
    #width_shift_range=0.2,
    #height_shift_range=0.2,
    # shear_range=0.2,
    #zoom_range=0.2,
    #horizontal_flip=True,
    # fill_mode='nearest'
    )

train_generator = train_datagen.flow_from_directory(
    training_dir,
    target_size=(300, 300),
    class_mode='categorical',
    # class_mode='sparse',
    shuffle=True,
    batch_size=16
)

test_datagen = ImageDataGenerator(rescale=1/255)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(300, 300),
    class_mode = 'categorical',
    # class_mode = 'sparse',
    shuffle=True,
    batch_size=16
)

X_pairs, y_pairs = generate_image_pairs(train_generator, num_pairs=1000)

# Ajustando o modelo com os dados gerados
# X_pairs[:, 0] e X_pairs[:, 1] são os pares de imagens
model.fit([X_pairs[:, 0], X_pairs[:, 1]], y_pairs, batch_size=16, epochs=10)

Found 100 images belonging to 6 classes.
Found 388 images belonging to 6 classes.
Epoch 1/10
63/63 [==============================] - 130s 2s/step - loss: 4.3200 - accuracy: 0.4630
Epoch 2/10
63/63 [==============================] - 131s 2s/step - loss: 4.3277 - accuracy: 0.4630
Epoch 3/10
63/63 [==============================] - 132s 2s/step - loss: 4.3277 - accuracy: 0.4630
Epoch 4/10
63/63 [==============================] - 132s 2s/step - loss: 4.3277 - accuracy: 0.4630
Epoch 5/10
63/63 [==============================] - 131s 2s/step - loss: 4.3277 - accuracy: 0.4630
Epoch 6/10
63/63 [==============================] - 131s 2s/step - loss: 4.3277 - accuracy: 0.4630
Epoch 7/10
63/63 [==============================] - 133s 2s/step - loss: 4.3277 - accuracy: 0.4630
Epoch 8/10
63/63 [==============================] - 134s 2s/step - loss: 4.3277 - accuracy: 0.4630
Epoch 9/10
62/63 [============================>.] - ETA: 2s - loss: 4.3463 - accuracy: 0.4607

KeyboardInterrupt: 